# Phase 1 · Reward benchmark (Kaggle or Colab, one T4 GPU)

Scores English→German WMT translations that experts rated (MQM) with each candidate reward, measures how well each reward agrees with the experts **when comparing translations of the same source**, probes each reward's blind spots, and suggests the training reward (R_train) and held-out evaluator (R_eval).
Expected time: **about 1–2 hours on one T4**. Details: [`docs/workflow.md` §4](https://github.com/guohere/machine-translation-with-RLHF/blob/claude/compassionate-ritchie-2nga86/docs/workflow.md).

### One-time setup
1. **Hugging Face:** open [Unbabel/wmt22-cometkiwi-da](https://huggingface.co/Unbabel/wmt22-cometkiwi-da) and accept the license (gated, non-commercial use). Create a *read* token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
2. **Kaggle:** right panel → *Session options* → Accelerator **GPU T4** · Internet **On** (needs a phone-verified account). *Add-ons → Secrets* → add `HF_TOKEN` and tick it for this notebook.
   **Colab:** *Runtime → Change runtime type* → **T4 GPU**. 🔑 *Secrets* panel → add `HF_TOKEN`, enable notebook access.
3. Run all cells. On Kaggle, **Save Version → Save & Run All (Commit)** runs in the background, so you can close the tab.

### When it finishes
Download **`p1_results.zip`** (Kaggle: *Output* tab of the saved version · Colab: *Files* panel) and either upload its contents to the repo under `results/p1/`, or paste `report/summary.md` into the chat.

In [ ]:
# ---- configuration ----
REPO = "https://github.com/guohere/machine-translation-with-RLHF.git"
BRANCH = "claude/compassionate-ritchie-2nga86"
TESTSETS = "wmt22 wmt23.sent"   # WMT en-de test sets with expert MQM scores
REWARDS = "chrf bleu cometkiwi22 comet22 metricx24_qe metricx24_ref"
RUN = "runs/p1"

In [ ]:
# ---- GPU check and Hugging Face token (works on Kaggle and Colab) ----
import os, subprocess

WORK = next((d for d in ("/kaggle/working", "/content") if os.path.isdir(d)), os.getcwd())
os.chdir(WORK)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE - switch the accelerator to a T4 GPU before continuing")

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

token = get_secret("HF_TOKEN")
if token:
    os.environ["HF_TOKEN"] = token
    print("HF_TOKEN loaded")
else:
    print("WARNING: no HF_TOKEN secret found - the gated CometKiwi model will fail to download")

In [ ]:
# ---- get the code and install (about 2 minutes) ----
if not os.path.isdir("machine-translation-with-RLHF"):
    !git clone -q -b {BRANCH} {REPO}
else:
    !git -C machine-translation-with-RLHF pull -q
os.chdir(os.path.join(WORK, "machine-translation-with-RLHF"))
!git log -1 --format="code version: %h %s"
# Note: this pins transformers<5 for this session (COMET 2.2.7 needs the transformers 4 output layout).
!pip install -q -e ".[phase1]"
# Installed without their declared dependencies on purpose (see pyproject.toml):
# unbabel-comet pins numpy<2 / transformers<5, mt-metrics-eval pulls apache_beam.
!pip install -q --no-deps unbabel-comet==2.2.7 "git+https://github.com/google-research/mt-metrics-eval"
!python -c "import torch, transformers, comet, mt_metrics_eval, mtrlhf; print('ready: torch', torch.__version__, '| transformers', transformers.__version__)"

### Step 1 · Download the WMT data and build the inputs (CPU, ~5–10 min)
Streams Google's mt-metrics-eval archive, keeps only the needed test sets, and writes `segments.jsonl` (rated translations) and `probes.jsonl` (blind-spot probes).

In [ ]:
!python scripts/p1_build_inputs.py --run {RUN} --testsets {TESTSETS} --download

### Step 2 · Smoke test (~3 min)
Scores 16 rows with the two neural QE rewards, to catch login or memory problems before the long run.

In [ ]:
!python scripts/p1_score.py --run {RUN} --rewards cometkiwi22 metricx24_qe --limit 16

### Step 3 · Score everything (~1–1.5 h on a T4)
Resumable: if the session stops, run this cell again and it continues where it left off.

In [ ]:
!python scripts/p1_score.py --run {RUN} --rewards {REWARDS}

### Step 4 · Analyze (CPU, ~2 min)

In [ ]:
!python scripts/p1_analyze.py --run {RUN} > /dev/null
from IPython.display import Markdown, Image, display
summary = open(f"{RUN}/report/summary.md").read().replace("![blind spots](blind_spots.png)", "")
display(Markdown(summary))
display(Image(f"{RUN}/report/blind_spots.png"))

In [ ]:
# ---- package the results for download ----
import shutil
pkg = os.path.join(WORK, "p1_results")
shutil.rmtree(pkg, ignore_errors=True)
shutil.copytree(f"{RUN}/report", f"{pkg}/report")
shutil.copytree(f"{RUN}/scores", f"{pkg}/scores")
shutil.copy(f"{RUN}/inputs_meta.json", pkg)
print("download:", shutil.make_archive(pkg, "zip", pkg))

### Troubleshooting
| Symptom | Fix |
|---|---|
| `401` / `403` / "gated repo" when loading `wmt22-cometkiwi-da` | Accept the license on the model page, and check the `HF_TOKEN` secret is attached to this notebook |
| "Temporary failure in name resolution" / downloads hang | Turn **Internet On** (Kaggle needs a phone-verified account) |
| `CUDA out of memory` while scoring MetricX | Rerun step 3 with `--max-tokens-per-batch 4096` added |
| An error inside `comet` mentioning `transformers` | Run `!pip install -q "transformers<5"`, restart the session, then run all cells again |
| The session stopped during step 3 | Run step 3 again; finished scores are kept and skipped |